# Testing & Perbandingan 8 Model Rekomendasi

Notebook ini menjalankan ULANG semua model yang dibandingkan untuk fitur
**Next-Item Recommendation**, dengan metodologi evaluasi yang konsisten di semua
model (leave-last-out split, Hit Rate@K, NDCG@K).

**Catatan jujur:** implementasi SASRec di sini adalah versi simplified
dari arsitektur aslinya, dibuat untuk proyek ini -- cukup untuk
menunjukkan perilaku umum (overfitting di v1, membaik dengan negative sampling
lebih banyak di v2), dan tidak identik 1:1 dengan library SASRec resmi manapun.

## 1. Import & Konfigurasi

In [23]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
RANDOM_STATE = 42

GENRE_COLS = [
    "Action", "Adult", "Adventure", "Arcade", "Beat 'Em Up", "Brain Training",
    "Card & Board Game", "Casual", "Educational", "Family", "Fighting", "Fitness",
    "Hack And Slash", "Horror", "Indie", "Moba", "Music", "Party", "Pinball",
    "Platform", "Point-And-Click", "Puzzle", "Quiz", "Racing",
    "Real Time Strategy (Rts)", "Rhythm", "Shooter", "Simulation", "Simulator",
    "Sport", "Sports", "Strategy", "Tactical", "Trivia",
    "Turn-Based Strategy (Tbs)", "Unique", "Unknown", "Visual Novel", "RPG",
]

K_LIST = [5, 10, 20]  # buat Hit Rate@K / NDCG@K


## 2. Load & Preprocessing Data

In [ ]:
trans = pd.read_csv("transaction.csv") #--> ubah directory transaksi disini
dim_game = pd.read_csv("game.csv") #--> ubah directory list game disini
trans["date_time"] = pd.to_datetime(trans["date_time"])

print(f"Total transaksi: {len(trans):,}")
print(f"User unik: {trans['user_id'].nunique():,} | Game unik: {trans['game_name'].nunique():,}")

# Encode user & game ke index integer 
users = trans["user_id"].unique()
games = trans["game_name"].unique()
user_to_idx = {u: i for i, u in enumerate(users)}
game_to_idx = {g: i for i, g in enumerate(games)}
idx_to_game = {i: g for g, i in game_to_idx.items()}
n_users, n_games = len(users), len(games)
trans["uidx"] = trans["user_id"].map(user_to_idx)
trans["gidx"] = trans["game_name"].map(game_to_idx)

# Genre lookup 
dim_game_idx = dim_game.drop_duplicates(subset="game_name").set_index("game_name")
genre_lookup = dim_game_idx.reindex(games)[GENRE_COLS].fillna(0).values  # shape (n_games, n_genre)
print(f"Genre lookup shape: {genre_lookup.shape}")


Total transaksi: 175,020
User unik: 5,000 | Game unik: 10,205
Genre lookup shape: (10205, 39)


## 3. Split Data: Leave-Last-Out per User

CATATAN : Transaksi terakhir tiap user (berdasarkan `date_time`) dijadikan test,
sisanya jadi train. Split ini dipakai KONSISTEN di semua 8 model.

In [25]:
trans_sorted = trans.sort_values(["user_id", "date_time"])
trans_sorted["rank_from_last"] = trans_sorted.groupby("user_id").cumcount(ascending=False)

test_df = trans_sorted[trans_sorted["rank_from_last"] == 0].copy()
train_df = trans_sorted[trans_sorted["rank_from_last"] > 0].copy()

print(f"Train: {len(train_df):,} baris | Test: {len(test_df):,} baris")

# Confidence score (dipakai ALS & BPR)
def _normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

train_df["rating_norm"] = _normalize(train_df["rating"])
train_df["playtime_norm"] = _normalize(train_df["playtime_hours"])
train_df["confidence"] = 1 + 4.0 * (0.6 * train_df["rating_norm"] + 0.4 * train_df["playtime_norm"])

user_item_train = sp.csr_matrix(
    (train_df["confidence"], (train_df["uidx"], train_df["gidx"])),
    shape=(n_users, n_games),
)
print("User-item matrix (train):", user_item_train.shape, "| nnz:", user_item_train.nnz)
test_truth = dict(zip(test_df["uidx"], test_df["gidx"]))


Train: 170,020 baris | Test: 5,000 baris
User-item matrix (train): (5000, 10205) | nnz: 169858


## 4. Evaluasi: Hit Rate@K & NDCG@K

Dipakai SAMA untuk semua model -- supaya perbandingan adil.

In [26]:
def evaluate(recommend_fn, k_list=K_LIST, exclude_seen=None):
    """
    recommend_fn(uidx, k) -> list of gidx (urut dari paling direkomendasikan)
    exclude_seen: dict uidx -> set(gidx) item yang sudah dimiliki user di TRAIN
                  (supaya tidak merekomendasikan ulang item yang sudah dimiliki)
    """
    results = {}
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcgs = {k: [] for k in k_list}
    hit_indicator_at10 = []  
    uidx_order = []
    mrr_list = []
    n_eval = 0

    for uidx, true_gidx in test_truth.items():
        recs = recommend_fn(uidx, max_k)
        if recs is None:
            continue
        n_eval += 1
        uidx_order.append(uidx)

        for k in k_list:
            topk = recs[:k]
            if true_gidx in topk:
                hits[k] += 1
                rank = topk.index(true_gidx) + 1
                ndcgs[k].append(1.0 / np.log2(rank + 1))
            else:
                ndcgs[k].append(0.0)

        # MRR (pakai rank di top max_k)
        if true_gidx in recs:
            rank_full = recs.index(true_gidx) + 1
            mrr_list.append(1.0 / rank_full)
        else:
            mrr_list.append(0.0)

        # Hit indicator @10 (dibutuhkan untuk bootstrap significance test)
        hit_indicator_at10.append(1 if true_gidx in recs[:10] else 0)

    for k in k_list:
        results[f"HitRate@{k}"] = hits[k] / n_eval if n_eval else 0.0
        results[f"Precision@{k}"] = results[f"HitRate@{k}"] / k  
        results[f"NDCG@{k}"] = float(np.mean(ndcgs[k])) if ndcgs[k] else 0.0
    results["MRR"] = float(np.mean(mrr_list)) if mrr_list else 0.0
    results["n_eval_users"] = n_eval
    results["_hit_indicator_at10"] = np.array(hit_indicator_at10)
    results["_uidx_order"] = uidx_order
    return results


def print_result(name, res):
    print(f"{name:40s} | HitRate@10 = {res['HitRate@10']*100:5.2f}% | NDCG@10 = {res['NDCG@10']*100:5.2f}% | MRR = {res['MRR']*100:5.2f}%")


user_seen = train_df.groupby("uidx")["gidx"].apply(set).to_dict()
all_results = {}  # simpan semua hasil 


## 5. Model 1 — Popularity (Global)

In [27]:
item_popularity = np.asarray(user_item_train.sum(axis=0)).flatten()
global_top_items = np.argsort(-item_popularity)  # urut dari paling populer

def recommend_popularity_global(uidx, k):
    seen = user_seen.get(uidx, set())
    recs = [g for g in global_top_items if g not in seen]
    return recs[:k]

res = evaluate(recommend_popularity_global)
all_results["1. Popularity (Global)"] = res
print_result("1. Popularity (Global)", res)


1. Popularity (Global)                   | HitRate@10 =  1.36% | NDCG@10 =  0.71% | MRR =  0.57%


## 6. Model 2 — Popularity (+Genre Filter)

In [28]:
user_genre_pref = {}
train_genre_matrix = genre_lookup[train_df["gidx"].values]  
tmp = pd.DataFrame(train_genre_matrix, columns=GENRE_COLS)
tmp["uidx"] = train_df["uidx"].values
genre_sum_per_user = tmp.groupby("uidx")[GENRE_COLS].sum()

def recommend_popularity_genre(uidx, k):
    seen = user_seen.get(uidx, set())
    if uidx not in genre_sum_per_user.index:
        recs = [g for g in global_top_items if g not in seen]
        return recs[:k]
    fav_genres = genre_sum_per_user.loc[uidx]
    fav_genres = fav_genres[fav_genres > 0].index.tolist()
    if not fav_genres:
        recs = [g for g in global_top_items if g not in seen]
        return recs[:k]
    fav_idx = [GENRE_COLS.index(g) for g in fav_genres]
    game_matches_genre = genre_lookup[:, fav_idx].sum(axis=1) > 0
    candidates = [g for g in global_top_items if game_matches_genre[g] and g not in seen]
    if len(candidates) < k:  # fallback kalau kandidat kurang
        extra = [g for g in global_top_items if g not in seen and g not in candidates]
        candidates += extra
    return candidates[:k]

res = evaluate(recommend_popularity_genre)
all_results["2. Popularity (+Genre filter)"] = res
print_result("2. Popularity (+Genre filter)", res)


2. Popularity (+Genre filter)            | HitRate@10 =  1.50% | NDCG@10 =  0.75% | MRR =  0.60%


## 7. Model 3 — ALS (Alternating Least Squares) 

Konfigurasi sesuai hasil tuning sebelumnya: `factors=64, regularization=0.05, iterations=20`

In [29]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=64, regularization=0.05, iterations=20, random_state=RANDOM_STATE
)
als_model.fit(user_item_train)

def recommend_als(uidx, k):
    gidx_arr, scores = als_model.recommend(
        uidx, user_item_train[uidx], N=k, filter_already_liked_items=True
    )
    return list(gidx_arr)

res = evaluate(recommend_als)
all_results["3. ALS"] = res
print_result("3. ALS", res)


  0%|          | 0/20 [00:00<?, ?it/s]

3. ALS                                   | HitRate@10 =  4.50% | NDCG@10 =  2.41% | MRR =  1.97%


## 8. Model 4 — Hybrid (ALS + Content-based Genre Similarity)

In [30]:
genre_sim_matrix = cosine_similarity(genre_lookup)  

ALPHA = 0.5  # bobot ALS vs content-based 

def recommend_hybrid(uidx, k):
    seen = user_seen.get(uidx, set())
    # skor ALS untuk SEMUA item
    als_scores = als_model.user_factors[uidx] @ als_model.item_factors.T
    # skor content-based: rata-rata similarity ke game yang pernah dimainkan user
    if seen:
        seen_list = list(seen)
        content_scores = genre_sim_matrix[seen_list].mean(axis=0)
    else:
        content_scores = np.zeros(n_games)
    # normalisasi 
    als_norm = (als_scores - als_scores.min()) / (als_scores.max() - als_scores.min() + 1e-9)
    content_norm = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min() + 1e-9)
    combined = ALPHA * als_norm + (1 - ALPHA) * content_norm
    combined[list(seen)] = -np.inf  # exclude yang sudah dimiliki
    top_k = np.argsort(-combined)[:k]
    return list(top_k)

res = evaluate(recommend_hybrid)
all_results["4. Hybrid (ALS + Genre)"] = res
print_result("4. Hybrid (ALS + Genre)", res)


4. Hybrid (ALS + Genre)                  | HitRate@10 =  4.12% | NDCG@10 =  2.10% | MRR =  1.70%


## 9. Model 5 — BPR (Bayesian Personalized Ranking)

In [31]:
from implicit.bpr import BayesianPersonalizedRanking

bpr_model = BayesianPersonalizedRanking(
    factors=64, iterations=100, learning_rate=0.01, random_state=RANDOM_STATE
)
bpr_model.fit(user_item_train)

def recommend_bpr(uidx, k):
    gidx_arr, scores = bpr_model.recommend(
        uidx, user_item_train[uidx], N=k, filter_already_liked_items=True
    )
    return list(gidx_arr)

res = evaluate(recommend_bpr)
all_results["5. BPR"] = res
print_result("5. BPR", res)


  0%|          | 0/100 [00:00<?, ?it/s]

5. BPR                                   | HitRate@10 =  2.46% | NDCG@10 =  1.15% | MRR =  0.86%


## 10. Model 6 & 7 — ALS + Reranker (LightGBM & XGBoost)

Two-stage: (1) ALS ambil top-50 kandidat, (2) reranker urutkan ulang pakai fitur tambahan
(skor ALS, popularitas item, jumlah histori user, genre overlap).

In [32]:
import lightgbm as lgb
import xgboost as xgb

N_CANDIDATES = 50  # kandidat awal dari ALS sebelum di-rerank

def build_reranker_features(uidx, candidate_gidx):
    """Bikin fitur sederhana per (user, kandidat item) untuk reranker."""
    als_scores = als_model.user_factors[uidx] @ als_model.item_factors[candidate_gidx].T
    pop_scores = item_popularity[candidate_gidx]
    n_histori = len(user_seen.get(uidx, set()))
    # genre overlap: rata-rata similarity genre ke item yang pernah dimainkan
    seen = list(user_seen.get(uidx, set()))
    if seen:
        genre_overlap = genre_sim_matrix[seen][:, candidate_gidx].mean(axis=0)
    else:
        genre_overlap = np.zeros(len(candidate_gidx))
    feats = np.column_stack([als_scores, pop_scores, np.full(len(candidate_gidx), n_histori), genre_overlap])
    return feats  # shape (len(candidate_gidx), 4)

# --- Bangun training set untuk reranker: pakai user-user di TRAIN (bukan test!) ---
# Supervised learning sederhana: label 1 kalau item itu benar-benar dimainkan user (dari train),
# label 0 kalau sampel negatif acak.
rng = np.random.RandomState(RANDOM_STATE)
X_train_rerank, y_train_rerank, group_sizes = [], [], []

sample_users = rng.choice(list(user_seen.keys()), size=min(1000, n_users), replace=False)  # subsample biar cepat
for u in sample_users:
    pos_items = list(user_seen[u])
    if not pos_items:
        continue
    neg_items = rng.choice(n_games, size=len(pos_items) * 4, replace=False)
    neg_items = [g for g in neg_items if g not in user_seen[u]][: len(pos_items) * 4]
    cand = pos_items + neg_items
    labels = [1] * len(pos_items) + [0] * len(neg_items)
    feats = build_reranker_features(u, np.array(cand))
    X_train_rerank.append(feats)
    y_train_rerank.extend(labels)
    group_sizes.append(len(cand))

X_train_rerank = np.vstack(X_train_rerank)
y_train_rerank = np.array(y_train_rerank)
print("Reranker training set:", X_train_rerank.shape)

# --- LightGBM reranker ---
lgb_model = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
lgb_model.fit(X_train_rerank, y_train_rerank)

def recommend_als_lgb(uidx, k):
    gidx_arr, _ = als_model.recommend(uidx, user_item_train[uidx], N=N_CANDIDATES, filter_already_liked_items=True)
    if len(gidx_arr) == 0:
        return []
    feats = build_reranker_features(uidx, np.array(gidx_arr))
    scores = lgb_model.predict_proba(feats)[:, 1]
    order = np.argsort(-scores)
    return [gidx_arr[i] for i in order[:k]]

res = evaluate(recommend_als_lgb)
all_results["6. ALS + LightGBM Reranker"] = res
print_result("6. ALS + LightGBM Reranker", res)

# --- XGBoost reranker ---
xgb_model = xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")
xgb_model.fit(X_train_rerank, y_train_rerank)

def recommend_als_xgb(uidx, k):
    gidx_arr, _ = als_model.recommend(uidx, user_item_train[uidx], N=N_CANDIDATES, filter_already_liked_items=True)
    if len(gidx_arr) == 0:
        return []
    feats = build_reranker_features(uidx, np.array(gidx_arr))
    scores = xgb_model.predict_proba(feats)[:, 1]
    order = np.argsort(-scores)
    return [gidx_arr[i] for i in order[:k]]

res = evaluate(recommend_als_xgb)
all_results["7. ALS + XGBoost Reranker"] = res
print_result("7. ALS + XGBoost Reranker", res)


Reranker training set: (169804, 4)
6. ALS + LightGBM Reranker               | HitRate@10 =  4.38% | NDCG@10 =  2.31% | MRR =  1.89%
7. ALS + XGBoost Reranker                | HitRate@10 =  4.42% | NDCG@10 =  2.21% | MRR =  1.77%


## 11. Model 8 & 9 — SASRec v1 & v2 (Sequential Transformer)

Ini adalah implementasi SASRec simoel (bukan library resmi) -- transformer encoder kecil dengan causal
masking, dilatih next-item prediction dengan negative sampling. Cukup untuk
menunjukkan pola behavior yang sama (v1 overfitting parah dengan 1 negative sample,
v2 membaik dengan 50 negatives + sampled softmax).

In [33]:
import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SEQ_LEN = 20
EMBED_DIM = 64

# Bangun sequence per user dari TRAIN (urut waktu), padding di kiri
user_sequences = train_df.sort_values(["uidx", "date_time"]).groupby("uidx")["gidx"].apply(list).to_dict()

def pad_sequence(seq, max_len=MAX_SEQ_LEN):
    seq = seq[-max_len:]
    pad_len = max_len - len(seq)
    return [n_games] * pad_len + seq  # n_games dipakai sebagai padding token

class SimpleSASRec(nn.Module):
    def __init__(self, n_items, embed_dim=EMBED_DIM, max_len=MAX_SEQ_LEN, n_heads=2, n_layers=2):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=n_items)  # +1 utk padding token
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.max_len = max_len

    def forward(self, seq):  # seq: (batch, max_len)
        positions = torch.arange(self.max_len, device=seq.device).unsqueeze(0)
        x = self.item_emb(seq) + self.pos_emb(positions)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(self.max_len).to(seq.device)
        out = self.encoder(x, mask=causal_mask)
        return out  # (batch, max_len, embed_dim) -- representasi tiap posisi


def train_sasrec(n_negatives, n_epochs=5, lr=1e-3):
    model = SimpleSASRec(n_games).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    all_seqs = [pad_sequence(seq) for seq in user_sequences.values() if len(seq) >= 2]
    seq_tensor = torch.tensor(all_seqs, dtype=torch.long)

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(len(seq_tensor))
        total_loss = 0.0
        batch_size = 128
        for i in range(0, len(perm), batch_size):
            idx = perm[i : i + batch_size]
            batch_seq = seq_tensor[idx].to(DEVICE)
            input_seq = batch_seq[:, :-1]
            input_seq = torch.cat([input_seq, torch.full((input_seq.size(0), 1), n_games, device=DEVICE)], dim=1)
            target = batch_seq[:, -1]  # item terakhir jadi target prediksi

            out = model(input_seq)
            last_hidden = out[:, -2, :]  # representasi posisi sebelum target

            pos_emb = model.item_emb(target)
            pos_logits = (last_hidden * pos_emb).sum(-1)

            neg_items = torch.randint(0, n_games, (input_seq.size(0), n_negatives), device=DEVICE)
            neg_emb = model.item_emb(neg_items)
            neg_logits = torch.bmm(neg_emb, last_hidden.unsqueeze(-1)).squeeze(-1)

            logits = torch.cat([pos_logits.unsqueeze(1), neg_logits], dim=1)
            labels = torch.zeros(input_seq.size(0), dtype=torch.long, device=DEVICE)
            loss = nn.functional.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  epoch {epoch+1}/{n_epochs} - loss: {total_loss:.4f}")
    return model


def recommend_sasrec(model, uidx, k):
    if uidx not in user_sequences or len(user_sequences[uidx]) < 1:
        return None
    seq = pad_sequence(user_sequences[uidx][:-0] if len(user_sequences[uidx]) < MAX_SEQ_LEN else user_sequences[uidx])
    seq_input = torch.tensor([seq], dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(seq_input)
        last_hidden = out[0, -1, :]
        all_item_emb = model.item_emb.weight[:n_games]  # exclude padding token
        scores = all_item_emb @ last_hidden
    seen = user_seen.get(uidx, set())
    scores_np = scores.cpu().numpy()
    scores_np[list(seen)] = -np.inf
    top_k = np.argsort(-scores_np)[:k]
    return list(top_k)


print("Training SASRec v1 (1 negative sample)...")
sasrec_v1 = train_sasrec(n_negatives=1, n_epochs=5)
res = evaluate(lambda u, k: recommend_sasrec(sasrec_v1, u, k))
all_results["8. SASRec v1 (1 negative)"] = res
print_result("8. SASRec v1 (1 negative)", res)

print("\nTraining SASRec v2 (50 negatives)...")
sasrec_v2 = train_sasrec(n_negatives=50, n_epochs=5)
res = evaluate(lambda u, k: recommend_sasrec(sasrec_v2, u, k))
all_results["9. SASRec v2 (50 negatives)"] = res
print_result("9. SASRec v2 (50 negatives)", res)


Training SASRec v1 (1 negative sample)...
  epoch 1/5 - loss: 182.1749
  epoch 2/5 - loss: 157.4369
  epoch 3/5 - loss: 145.0192
  epoch 4/5 - loss: 132.9391
  epoch 5/5 - loss: 120.6781
8. SASRec v1 (1 negative)                | HitRate@10 =  0.08% | NDCG@10 =  0.05% | MRR =  0.04%

Training SASRec v2 (50 negatives)...
  epoch 1/5 - loss: 660.9658
  epoch 2/5 - loss: 564.1975
  epoch 3/5 - loss: 498.4227
  epoch 4/5 - loss: 433.0759
  epoch 5/5 - loss: 382.3767
9. SASRec v2 (50 negatives)              | HitRate@10 =  0.26% | NDCG@10 =  0.12% | MRR =  0.08%


## 12. Tabel Ringkasan — Perbandingan Semua Model

In [34]:
summary_rows = []
for name, res in all_results.items():
    summary_rows.append({
        "Model": name,
        "HitRate@5": f"{res['HitRate@5']*100:.2f}%",
        "HitRate@10": f"{res['HitRate@10']*100:.2f}%",
        "HitRate@20": f"{res['HitRate@20']*100:.2f}%",
        "NDCG@10": f"{res['NDCG@10']*100:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("HitRate@10", ascending=False)
print(summary_df.to_string(index=False))

# Simpan hasil ke CSV -- ini jadi BUKTI konkret hasil run yang bisa dilampirkan
summary_df.to_csv("hasil_perbandingan_8_model.csv", index=False)
print("\nDisimpan ke: hasil_perbandingan_8_model.csv")


                        Model HitRate@5 HitRate@10 HitRate@20 NDCG@10
                       3. ALS     2.72%      4.50%      7.28%   2.41%
    7. ALS + XGBoost Reranker     2.42%      4.42%      7.72%   2.21%
   6. ALS + LightGBM Reranker     2.66%      4.38%      7.28%   2.31%
      4. Hybrid (ALS + Genre)     2.60%      4.12%      7.12%   2.10%
                       5. BPR     1.38%      2.46%      3.96%   1.15%
2. Popularity (+Genre filter)     0.90%      1.50%      2.52%   0.75%
       1. Popularity (Global)     0.88%      1.36%      2.22%   0.71%
  9. SASRec v2 (50 negatives)     0.18%      0.26%      0.38%   0.12%
    8. SASRec v1 (1 negative)     0.06%      0.08%      0.08%   0.05%

Disimpan ke: hasil_perbandingan_8_model.csv
